# Symbolic verification of the Prox-ITEM Lyapunov simplification

This notebook verifies the algebraic identity used in the Prox-ITEM Lyapunov proof:

$$
\mathcal S_k
=p_0\left\langle z^k-x^k,
L\delta_k^{-1}(z^k-z^{k+1})+qL(x^\star-z^k)+s_g^\star-s_g^{k+1}
\right\rangle
+\langle z^k-z^{k+1},\eta^k\rangle,
$$

where

$$
\eta^k=p_1\delta_k^{-2}\big(L(x^\star-z^k)+\delta_k(s_g^{k+1}-s_g^\star)\big)
+p_2\delta_k^{-1}(s_g^k-s_g^\star)
+p_3\delta_k^{-2}L(z^k+z^{k+1}-2x^\star),
$$

and

$$
\begin{aligned}
p_0
&=A_k-(1-q)A_{k+1}\beta_k,\\
p_1
&=\delta_k\left(A_k-(1+q)A_{k+1}\right)+2A_{k+1},\\
p_2
&=\delta_k(\sigma_k-1)-A_k,\\
p_3
&=A_{k+1}-(1+qA_{k+1})\delta_k^2.
\end{aligned}
$$

It also verifies that the Prox-ITEM parameter choices imply $p_0=p_1=p_2=p_3=0$.

The notebook is organized as follows:

1. define symbols matching the notation used in the paper;
2. transcribe $\mathcal S_k$ exactly in those symbols;
3. apply the update-rule substitutions used for the direct simplification;
4. transcribe the target expression with the polynomial coefficients $p_0,p_1,p_2,p_3$;
5. let SymPy reduce the difference to zero;
6. verify that the Prox-ITEM parameter choices make each $p_i$ vanish.

We work in one scalar coordinate. Since the identity is a finite linear combination of inner products and squared norms with scalar coefficients, this scalar identity proves the Hilbert-space identity coordinatewise.


In [ ]:
import sympy as sp

sp.init_printing()


def inner(u, v):
    return u * v


def sq(expr):
    return expr**2


def norm2(v):
    return sq(v)


## 1. Symbols matching the proof

The table fixes the translation from the notation used in the proof to Python identifiers. These are formal symbols; no update rules are applied in this section.

| Notation | Python identifier |
|---|---|
| $A_k$, $A_{k+1}$ | `A_k`, `A_kp1` |
| $q$, $L$, $\mu$ | `q`, `L`, `mu` |
| $\beta_k$, $\delta_k$, $\sigma_k$ | `beta_k`, `delta_k`, `sigma_k` |
| $z^k$, $z^{k+1}$ | `z_k`, `z_kp1` |
| $x^k$, $x^\star$ | `x_k`, `x_star` |
| $y^k$, $y^{k-1}$ | `y_k`, `y_km1` |
| $s_g^k$, $s_g^{k+1}$, $s_g^\star$ | `s_g_k`, `s_g_kp1`, `s_g_star` |
| $\nabla f(y^k)$, $\nabla f(y^{k-1})$, $\nabla f(x^\star)$ | `grad_f_y_k`, `grad_f_y_km1`, `grad_f_x_star` |


In [ ]:
A_k, A_kp1, q, L, mu = sp.symbols("A_k A_{k+1} q L mu", nonzero=True)
beta_k, delta_k, sigma_k = sp.symbols("beta_k delta_k sigma_k", nonzero=True)

z_k, z_kp1 = sp.symbols("z_k z_{k+1}")
x_k, x_star = sp.symbols("x_k x_star")
y_k, y_km1 = sp.symbols("y_k y_{k-1}")

s_g_k, s_g_kp1, s_g_star = sp.symbols("s_g^k s_g^{k+1} s_g^star")
grad_f_y_k, grad_f_y_km1, grad_f_x_star = sp.symbols(
    "grad_f_y_k grad_f_y_{k-1} grad_f_x_star"
)


## 2. The displayed expression for $\mathcal S_k$

The following cell transcribes $\mathcal S_k$ after function-value cancellation and after adding the two norm terms to both sides of the inequality.

No update-rule substitutions are used here.


In [ ]:
added_norm_terms = (
    (A_kp1 - A_k) / (2 * L) * norm2(s_g_kp1 - s_g_star)
    + A_k / (2 * L) * norm2(s_g_k - s_g_kp1)
)

smooth_xstar_yk_terms = (
    -(1 - q) * (A_kp1 - A_k)
    * (
    inner(grad_f_y_k, x_star - y_k) + mu / 2 * norm2(x_star - y_k)
    )
    - ((1 - q) * (A_kp1 - A_k)) / (2 * (L - mu))
    * norm2(
    grad_f_x_star - grad_f_y_k - mu * (x_star - y_k)
    )
)

smooth_ykm1_yk_terms = (
    -(1 - q) * A_k
    * (
    inner(grad_f_y_k, y_km1 - y_k) + mu / 2 * norm2(y_km1 - y_k)
    )
    - ((1 - q) * A_k) / (2 * (L - mu))
    * norm2(
    grad_f_y_km1 - grad_f_y_k - mu * (y_km1 - y_k)
    )
)

smooth_yk_xstar_terms = (
    -(1 - q) * A_kp1
    * (
    inner(grad_f_x_star, y_k - x_star) + mu / 2 * norm2(y_k - x_star)
    )
    - ((1 - q) * A_kp1) / (2 * (L - mu))
    * norm2(
    grad_f_y_k - grad_f_x_star - mu * (y_k - x_star)
    )
)

smooth_ykm1_xstar_terms = (
    +(1 - q) * A_k
    * (
    inner(grad_f_x_star, y_km1 - x_star) + mu / 2 * norm2(y_km1 - x_star)
    )
    + ((1 - q) * A_k) / (2 * (L - mu))
    * norm2(
    grad_f_y_km1 - grad_f_x_star - mu * (y_km1 - x_star)
    )
)

g_inner_product_terms = (
    -((1 + q) * A_kp1 - A_k) * inner(s_g_kp1, x_star - z_kp1)
    + q * A_k * inner(s_g_k, x_star - z_k)
    - ((1 + q) * A_kp1 - A_k - sigma_k + 1)
    * inner(
    s_g_star, z_kp1 - x_star
    )
    + (q * A_k + 1 - sigma_k) * inner(s_g_star, z_k - x_star)
    - (sigma_k - 1) * inner(s_g_k, z_kp1 - z_k)
)

subgradient_norm_terms = (
    -A_k / (2 * L) * norm2(s_g_k - s_g_star)
    + A_kp1 / (2 * L) * norm2(s_g_kp1 - s_g_star)
)

distance_terms = (
    (L + mu * A_kp1) * norm2(z_kp1 - x_star)
    - (L + mu * A_k) * norm2(z_k - x_star)
)

S_displayed = sum(
    [
        added_norm_terms,
        smooth_xstar_yk_terms,
        smooth_ykm1_yk_terms,
        smooth_yk_xstar_terms,
        smooth_ykm1_xstar_terms,
        g_inner_product_terms,
        subgradient_norm_terms,
        distance_terms,
    ],
    sp.Rational(0),
)


## 3. Substitutions used by SymPy

We now apply the update rules and definitions listed in the proof text for the direct simplification. No parameter formula is substituted for $A_{k+1}$, $\beta_k$, $\delta_k$, or $\sigma_k$ in this step. These four symbols remain free in the resulting coefficients. The recursion defining $A_{k+1}$ and the definitions of $\beta_k$, $\delta_k$, and $\sigma_k$ are used later, only for the coefficient cancellation check. This gives the substitutions

$$
\mu=qL,\qquad
y^k=(1-\beta_k)z^k+\beta_k x^k,
$$

$$
\nabla f(y^k)
=L\delta_k^{-1}(z^k-z^{k+1})+qL(y^k-z^k)-s_g^{k+1},
$$

$$
\nabla f(y^{k-1})=L(y^{k-1}-x^k)-s_g^k,
\qquad
\nabla f(x^\star)=-s_g^\star.
$$

The scalar substitutions for $\mu$, $y^k$, and $\nabla f(x^\star)$ are copied directly from initialization, the extrapolation formula, and optimality:

$$
q=\mu L^{-1},
\qquad
\nabla f(x^\star)+s_g^\star=0.
$$

For the gradient substitution, start with the $\bar z$-update and sampled-subgradient,

$$
\bar z^{k+1}
=(1-q\delta_k)z^k+q\delta_k y^k-
\delta_k L^{-1}\nabla f(y^k),
\qquad
s_g^{k+1}=L\delta_k^{-1}(\bar z^{k+1}-z^{k+1}).
$$

The sampled-subgradient gives $\bar z^{k+1}=z^{k+1}+\delta_k L^{-1}s_g^{k+1}$. Substituting this into the $\bar z$-update gives

$$
z^{k+1}+\delta_k L^{-1}s_g^{k+1}
=(1-q\delta_k)z^k+q\delta_k y^k-
\delta_k L^{-1}\nabla f(y^k).
$$

Solving for $\nabla f(y^k)$ gives

$$
\nabla f(y^k)
=L\delta_k^{-1}(z^k-z^{k+1})+qL(y^k-z^k)-s_g^{k+1}.
$$

For the previous-step gradient, use the $x$-update at the previous index and the sampled-subgradient definition,

$$
x^k
=y^{k-1}-L^{-1}\nabla f(y^{k-1})
-\delta_{k-1}^{-1}(\bar z^k-z^k),
\qquad
s_g^k=L\delta_{k-1}^{-1}(\bar z^k-z^k).
$$

These two equations give

$$
x^k=y^{k-1}-L^{-1}\left(\nabla f(y^{k-1})+s_g^k\right),
$$

and therefore

$$
\nabla f(y^{k-1})=L(y^{k-1}-x^k)-s_g^k.
$$

The list `update_substitutions` is intentionally explicit: it is the bridge between the displayed expression and the expression that SymPy reduces.


In [ ]:
y_k_substitution = (1 - beta_k) * z_k + beta_k * x_k

grad_f_y_k_substitution = (
    L * delta_k**-1 * (z_k - z_kp1)
    + q * L * (y_k - z_k)
    - s_g_kp1
)
grad_f_y_km1_substitution = L * (y_km1 - x_k) - s_g_k
grad_f_x_star_substitution = -s_g_star

update_substitutions = [
    (grad_f_y_k, grad_f_y_k_substitution),
    (grad_f_y_km1, grad_f_y_km1_substitution),
    (grad_f_x_star, grad_f_x_star_substitution),
    (y_k, y_k_substitution),
    (mu, q * L),
]

S_after_substitutions = S_displayed.subs(update_substitutions, simultaneous=False)


## 4. Target expression

The target expression is

$$
p_0\left\langle z^k-x^k,
L\delta_k^{-1}(z^k-z^{k+1})+qL(x^\star-z^k)+s_g^\star-s_g^{k+1}
\right\rangle
+\langle z^k-z^{k+1},\eta^k\rangle,
$$

where

$$
\eta^k=p_1\delta_k^{-2}\big(L(x^\star-z^k)+\delta_k(s_g^{k+1}-s_g^\star)\big)
+p_2\delta_k^{-1}(s_g^k-s_g^\star)
+p_3\delta_k^{-2}L(z^k+z^{k+1}-2x^\star).
$$

The four scalar coefficients are the polynomials

$$
p_0=A_k-(1-q)A_{k+1}\beta_k,
$$

$$
p_1=\delta_k\left(A_k-(1+q)A_{k+1}\right)+2A_{k+1},
$$

$$
p_2=\delta_k(\sigma_k-1)-A_k,
$$

$$
p_3=A_{k+1}-(1+qA_{k+1})\delta_k^2.
$$

In [ ]:
p_0 = A_k - (1 - q) * A_kp1 * beta_k
p_1 = delta_k * (A_k - (1 + q) * A_kp1) + 2 * A_kp1
p_2 = delta_k * (sigma_k - 1) - A_k
p_3 = A_kp1 - (1 + q * A_kp1) * delta_k**2

extra_target_term = p_0 * (z_k - x_k) * (
    L * delta_k**-1 * (z_k - z_kp1)
    + q * L * (x_star - z_k)
    + s_g_star
    - s_g_kp1
)

eta_new = (
    p_1 * delta_k**-2 * (L * (x_star - z_k) + delta_k * (s_g_kp1 - s_g_star))
    + p_2 * delta_k**-1 * (s_g_k - s_g_star)
    + p_3 * delta_k**-2 * L * (z_k + z_kp1 - 2 * x_star)
)
target_new = extra_target_term + (z_k - z_kp1) * eta_new


## 5. SymPy reduction

Both sides are now rational expressions in the same symbols. SymPy clears denominators and reduces the difference between the substituted raw slack and the target expression to zero.


In [ ]:
slack_identity_residual = sp.factor(
    sp.together(sp.expand(S_after_substitutions - target_new))
)

assert slack_identity_residual == 0
slack_identity_residual


## 6. Coefficient cancellation check

This section checks the final step of the proof. We use the parameter identities

$$
A_{k+1}=\left((1+q)A_k+2(1+\sigma_k)\right)(1-q)^{-2},
\qquad
\beta_k=A_k\left((1-q)A_{k+1}\right)^{-1},
$$

$$
\delta_k^2=A_{k+1}(1+qA_{k+1})^{-1},
\qquad
\sigma_k^2=(1+A_k)(1+qA_k).
$$

The checks below follow the paper text. First, $p_0=0$ uses the definition of $\beta_k$. Next, $p_1=p_2=0$ uses the definition of $\delta_k$, the recursion for $A_{k+1}$, and the definition of $\sigma_k$. Finally, $p_3=0$ uses only the definition of $\delta_k^2$.

In [ ]:
A_kp1_recursion = ((1 + q) * A_k + 2 * (1 + sigma_k)) / (1 - q)**2
beta_definition = A_k / ((1 - q) * A_kp1)
delta_squared_definition = A_kp1 / (1 + q * A_kp1)
sigma_squared_definition = (1 + A_k) * (1 + q * A_k)


### The coefficient $p_0$

The coefficient $p_0$ uses only the definition of $\beta_k$.

In [ ]:
p_0_after_beta = sp.factor(p_0.subs(beta_k, beta_definition))

assert p_0_after_beta == 0
p_0_after_beta


### The coefficient $p_1$

The coefficient can be written as

$$
p_1=2A_{k+1}-\delta_k\left((1+q)A_{k+1}-A_k\right).
$$

Thus $p_1=0$ follows from

$$
\delta_k^2\left((1+q)A_{k+1}-A_k\right)^2=4A_{k+1}^2,
$$

together with $\delta_k>0$, $A_{k+1}>0$, and $(1+q)A_{k+1}-A_k>0$.


In [ ]:
p_1_squared_residual = (
    delta_k**2 * ((1 + q) * A_kp1 - A_k) ** 2 - 4 * A_kp1**2
)

p_1_squared_residual = p_1_squared_residual.subs(
    delta_k**2, delta_squared_definition
)
p_1_squared_residual = p_1_squared_residual.subs(A_kp1, A_kp1_recursion)
p_1_squared_residual = sp.factor(p_1_squared_residual)
p_1_check = p_1_squared_residual.subs(
    sigma_k**2, sigma_squared_definition
)
p_1_check = sp.factor(p_1_check)

assert p_1_check == 0

p_1_check


### The coefficient $p_2$

The coefficient can be written as

$$
p_2=\delta_k(\sigma_k-1)-A_k.
$$

Thus $p_2=0$ follows from

$$
\delta_k^2(\sigma_k-1)^2=A_k^2,
$$

together with $\delta_k>0$, $\sigma_k\ge 1$, and $A_k\ge 0$.

In [ ]:
p_2_squared_residual = delta_k**2 * (sigma_k - 1) ** 2 - A_k**2

p_2_squared_residual = p_2_squared_residual.subs(
    delta_k**2, delta_squared_definition
)
p_2_squared_residual = p_2_squared_residual.subs(A_kp1, A_kp1_recursion)
p_2_squared_residual = sp.factor(p_2_squared_residual)
p_2_check = p_2_squared_residual.subs(
    sigma_k**2, sigma_squared_definition
)
p_2_check = sp.factor(p_2_check)

assert p_2_check == 0

p_2_check


### The coefficient $p_3$

The coefficient $p_3$ uses only the definition of $\delta_k^2$.

In [ ]:
p_3_after_delta = sp.factor(p_3.subs(delta_k**2, delta_squared_definition))

assert p_3_after_delta == 0
p_3_after_delta
